# EPF (padronizado)


*By Miguel Ferreira*
**Este notebook, assim com todos os outros de cada ferramenta do envelope de risco, segue o mesmo protocolo.**


## 1) Importação do dataset e bibliotecas


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

sys.path.append(str(Path("../../src").resolve()))
from setup import setup

CSV_PATH = "../../data/processed/financial_tools_datset.csv"
TARGET_COL = "Price"
HORIZON = 1


## 2) Execução do `setup()` e alinhamento temporal


In [ ]:
raw_df = pd.read_csv(CSV_PATH)
raw_df["Date"] = pd.to_datetime(raw_df["Date"], format="%m/%d/%Y")
raw_df = raw_df.sort_values("Date").set_index("Date")

splits = setup(
    csv_path=CSV_PATH,
    target_col=TARGET_COL,
    horizon=HORIZON,
)

n_rows = len(raw_df)
train_end = len(splits["X_train"])
val_end = train_end + len(splits["X_val"])
val_index = raw_df.iloc[train_end:val_end].index

print("Train shape:", splits["X_train"].shape)
print("Val shape:", splits["X_val"].shape)
print("Test shape:", splits["X_test"].shape)


## 3) Construção da ferramenta de risco


In [ ]:
y_train_bin = (splits["y_train"] > 0).astype(int)
y_val_bin = (splits["y_val"] > 0).astype(int)

model = LogisticRegression(max_iter=1000)
model.fit(splits["X_train"], y_train_bin)

val_probs = model.predict_proba(splits["X_val"])[:, 1]
auc = roc_auc_score(y_val_bin, val_probs)
print("Validation AUC:", auc)

epf_probs = pd.Series(val_probs, index=val_index, name="epf_prob")
epf_score = epf_probs.rolling(50).mean()

price = raw_df.loc[val_index, TARGET_COL]
epf_upper = price * (1 + epf_score)
epf_lower = price * (1 - epf_score)

epf_dataset = pd.DataFrame(index=val_index)
epf_dataset["epf_upper"] = epf_upper
epf_dataset["epf_lower"] = epf_lower
epf_dataset["epf_width"] = epf_dataset["epf_upper"] - epf_dataset["epf_lower"]
mid = (epf_dataset["epf_upper"] + epf_dataset["epf_lower"]) / 2
epf_dataset["epf_asymmetry"] = (price - mid) / epf_dataset["epf_width"]


## 4) Sanity checks mínimos


In [ ]:
print("NaN críticos:")
print(epf_dataset.isna().sum())

print("\nAlinhamento temporal:")
print("Index monotonic increasing:", epf_dataset.index.is_monotonic_increasing)
print("Index has duplicates:", epf_dataset.index.has_duplicates)
print("Dataset lines == val_index lines:", len(epf_dataset) == len(val_index))


## 5) DataFrame final (features de risco)


In [ ]:
epf_dataset = epf_dataset[["epf_upper", "epf_lower", "epf_width", "epf_asymmetry"]]
epf_dataset.head(), epf_dataset.shape


## 6) Padronização (nome de colunas, dtype, index)


In [ ]:
epf_dataset = epf_dataset.sort_index().bfill()
epf_dataset.index.name = "timestamp"
epf_dataset = epf_dataset.astype(np.float32)

epf_dataset.head(), epf_dataset.dtypes


## 7) Salvamento em `.parquet`


In [ ]:
epf_dataset.to_parquet("epf_features.parquet", index=True)
